# 01 — Perfilado inicial (Entrega 1)

Objetivo: verificar volúmenes, tipos, nulos, cardinalidad e integridad referencial de las dos fuentes del proyecto. Las cifras de este notebook alimentan `docs/data_dictionary.md`.

El perfilado **formal y completo** (con reglas de limpieza y homologación) corresponde a la Entrega 2.

In [ ]:
import pandas as pd

catalog = pd.read_csv('../data/raw/catalog_raw.csv')
movements = pd.read_csv('../data/raw/movements_raw.csv', parse_dates=['timestamp'])

print(f'Catálogo:    {catalog.shape[0]} filas × {catalog.shape[1]} columnas')
print(f'Movimientos: {movements.shape[0]} filas × {movements.shape[1]} columnas')

## Catálogo — tipos, nulos y cardinalidad

In [ ]:
pd.DataFrame({
    'dtype': catalog.dtypes.astype(str),
    'nulos': catalog.isnull().sum(),
    'cardinalidad': catalog.nunique(),
})

In [ ]:
# Distribución de nutriscore y categorías
print('Nutriscore:', catalog['nutriscore'].value_counts().to_dict())
print(f"Categorías distintas: {catalog['category'].nunique()}")

In [ ]:
# Detección del outlier en calories_100g (R2 de la propuesta)
outliers = catalog[catalog['calories_100g'] > 900]
outliers[['product_id', 'product_name', 'calories_100g']]

## Movimientos — tipos, nulos y cardinalidad

In [ ]:
pd.DataFrame({
    'dtype': movements.dtypes.astype(str),
    'nulos': movements.isnull().sum(),
    'cardinalidad': movements.nunique(),
})

In [ ]:
print('Rango temporal:', movements['timestamp'].min(), '→', movements['timestamp'].max())
print('Acciones: ', movements['action_type'].value_counts().to_dict())
print('Ubicaciones:', movements['location'].value_counts().to_dict())

## Integridad referencial

In [ ]:
huerfanos = (~movements['product_id'].astype(str).isin(catalog['product_id'].astype(str))).sum()
print(f'Eventos con product_id ausente en el catálogo: {huerfanos}')

## Nulos estructurales en `expiry_date`

Se espera que `expiry_date` esté vacío en los eventos `OUT` y presente en los `IN`. Verificación:

In [ ]:
movements.groupby('action_type')['expiry_date'].apply(lambda s: s.isnull().sum())